# TetraFT — Kaggle runs (heal / polish / α·T scout)

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (**refresh** after code changes)
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`
- **B only:** Session A full `checkpoint-final`
- **P only:** Session B `checkpoint-final` (weights-only OK)
- **S (α/T scout):** code + FineWeb only — **fresh start, no ckpt**

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run |

| SESSION | Preset | Resume | Gate |
|---------|--------|--------|------|
| **A** | `heal_kl_50m` | none → 6104 | mid ≲50–52 |
| **B** | `heal_kl_50m` | A full → 12207 | &lt;43.77 (done ~34.38) |
| **P** | `polish_kl_5m` | B → 13487 | &lt;34.38 (**FAIL** — stop polish) |
| **S** | `scout_kl_5m` + α/T | **fresh** 1280 | **&lt;49.31** |

**Current next:** `SESSION="S"`, first cell **α=0.3 T=2.0** (`a03_t2`).  
Logic in `run_smoke.py` — notebook is glue only.


In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

from config import SMOKE_PRESETS
assert "heal_kl_50m" in SMOKE_PRESETS, "heal_kl_50m missing — refresh tetraft-code"
assert "polish_kl_5m" in SMOKE_PRESETS, "polish_kl_5m missing — refresh tetraft-code"
assert "scout_kl_5m" in SMOKE_PRESETS
print("presets ok:", sorted(SMOKE_PRESETS))

In [ ]:
from run_smoke import run_smoke
import argparse
import shutil
from pathlib import Path

# =============================================================================
# SESSION: "A" | "B" | "P" | "S" (α/T scout — current next)
# =============================================================================
SESSION = "S"  # <-- A | B | P | S

# SESSION=S only: which distill cell (baseline locked α=0.5 T=2 → 49.31)
SCOUT_ALPHA = 0.3          # 0.3 = more teacher; 0.5 = baseline
SCOUT_TEMPERATURE = 2.0    # 2.0 soft; 1.0 harder KL
SCOUT_TAG = "a03_t2"       # output dir tag; match α/T (a03_t2 | a05_t1 | a03_t1)

CLEAR_OUTPUT = True

if SESSION.upper() == "A":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 6104
    SAVE_OPTIMIZER = True
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_A"
elif SESSION.upper() == "B":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 12207
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_B"
    print("resume from:", RESUME)
elif SESSION.upper() == "P":
    PRESET = "polish_kl_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_polish_kl_5m"
    print("polish resume from:", RESUME)
elif SESSION.upper() == "S":
    # Fresh α/T scout @ ~5.24M — match scout_kl_5m DNA; only α/T change
    PRESET = "scout_kl_5m"
    MAX_STEPS = None  # preset 1280
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = float(SCOUT_ALPHA)
    DISTILL_TEMPERATURE = float(SCOUT_TEMPERATURE)
    OUTPUT_DIR = f"/kaggle/working/checkpoints_scout_kl_{SCOUT_TAG}"
    print(f"α/T scout: α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE} tag={SCOUT_TAG}")
    print("gate: end PPL < 49.31 (locked scout_kl_5m)")
else:
    raise ValueError("SESSION must be 'A', 'B', 'P', or 'S'")

out = Path(OUTPUT_DIR)
if CLEAR_OUTPUT and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=MAX_STEPS,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=SKIP_SHOCK,
    skip_orig=SKIP_ORIG,
    resume=RESUME,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    schedule_max_steps=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    save_optimizer=SAVE_OPTIMIZER,
    skip_linear_attn=None,
    no_skip_linear_attn=False,
    distill_alpha=DISTILL_ALPHA,
    distill_temperature=DISTILL_TEMPERATURE,
    quant_reg_beta=None,
    seed=42,
    device_map="auto",
)
print(
    f"SESSION={SESSION} preset={PRESET} max_steps={MAX_STEPS} "
    f"α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE} resume={RESUME} out={OUTPUT_DIR}"
)
print("note: KL loads frozen FP teacher (~2× VRAM)")
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
    "resumed_step", "schedule_horizon_steps", "distill",
]
print({k: results[k] for k in keys if k in results})
if "inventory_summary" in results:
    inv = results["inventory_summary"]
    print("inventory", inv)
    if inv.get("n_eligible", 0) > 150:
        print("WARNING: eligible looks like all-Linear — GDN skip may be off")
ppl = results.get("ppl_after_smoke")
if ppl is not None:
    ref = results.get("ppl_original") or results.get("ppl_original_ref") or 17.67
    print(f"after/orig ≈ {ppl / ref:.3f} (ref orig {ref})")
    if SESSION.upper() == "A":
        print(f"Session A mid PPL={ppl:.2f} — go/no-go: continue B if ≲50–52 and falling")
    elif SESSION.upper() == "B":
        print(f"Session B final PPL={ppl:.2f} — bar CE heal_50m ~43.77; strong if ≲35")
    elif SESSION.upper() == "P":
        print(f"Polish final PPL={ppl:.2f} — gate < 34.38")
    else:
        gate = 49.31
        print(f"α/T scout final PPL={ppl:.2f} — gate < {gate} (scout_kl_5m)")
        if ppl < gate:
            print(f"PASS — lock α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE} for next long KL")
            print("Do NOT re-polish old B only; fresh heal_kl with winning α/T")
        else:
            print("NO PASS — try next cell (a05_t1) or keep α=0.5 T=2; see RESULTS.md §5.4")

### α/T scout cells (SESSION=S)

| Tag | α | T | When |
|-----|---|---|------|
| baseline done | 0.5 | 2.0 | **49.31** gate |
| **`a03_t2`** (first) | **0.3** | 2.0 | more teacher |
| `a05_t1` | 0.5 | **1.0** | harder KL |
| `a03_t1` | 0.3 | 1.0 | only if both help |

Set `SCOUT_ALPHA`, `SCOUT_TEMPERATURE`, `SCOUT_TAG` to match. One cell per job.

### Frozen baselines

| Run | ≈ tokens | Val PPL |
|-----|---------:|--------:|
| Original FP | — | ~17.7 |
| scout_kl_5m | 5.2M | **~49.31** |
| CE heal_50m | 50M | ~43.77 |
| heal_kl_50m A+B | 50M | **~34.38** |
| polish_kl_5m | +5.2M | FAIL (≥34.38) — stop polish |

Record scout PPL in `RESULTS.md`. Future Muon 5M: `RESULTS.md` §5.7 (not implemented).
